# Project 2 — LLM Stress Testing & Edge Case Benchmark (Turing Portfolio)

**40 challenging prompts** — 10 Software Engineering, 10 Programming, 10 ERP, 10 Requirements Testing. Notebook ini mencakup *expected answers*, respons dari LLM yang **benar-benar dijalankan**, deteksi awal indikasi kesalahan, klasifikasi, saran perbaikan, human review, perbandingan model, grafik dan ekspor CSV/JSON/ZIP.

> **Integritas evaluasi:** `required_patterns` adalah *heuristic concept coverage*, bukan ground-truth accuracy. Respons yang tidak menyebut istilah tertentu belum tentu salah; respons yang menyebut semua istilah belum tentu benar. Flag pelanggaran `prohibited_patterns` adalah sinyal yang harus diverifikasi manusia. Data `demo` dilabeli **SIMULATED**, bukan respons model. Tidak ada angka kinerja model diklaim sebelum inferensi berjalan.

**Lingkungan:** Google Colab, GPU T4 disarankan untuk full benchmark. Untuk mencoba cepat, jalankan 8 kasus dahulu. Model default `Qwen/Qwen2.5-0.5B-Instruct` sebagai *small baseline*, bukan standar kualitas frontier.

## 1. Setup
Gunakan **Runtime → Change runtime type → T4 GPU** bila tersedia. Jalankan sel secara berurutan. Perlu internet untuk mengunduh model dari Hugging Face.

In [ ]:
%pip -q install "transformers>=4.45,<5" "accelerate>=0.33" "pandas>=2.0" "matplotlib>=3.7" "scikit-learn>=1.3" "ipywidgets>=8.0" "sentencepiece>=0.2"

## 2. Konfigurasi eksperimen
`RUN_MODE='live'` menjalankan model sungguhan. `RUN_MODE='demo'` hanya untuk tes alur tanpa model; **jangan laporkan demo sebagai hasil benchmark LLM**. Untuk 40 kasus ubah `MAX_CASES = 40`. `MODEL_IDS` dapat berisi lebih dari satu model (butuh memori dan waktu lebih).

In [ ]:
from pathlib import Path
import os, re, json, gc, time, random, hashlib, zipfile, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

RUN_MODE = 'live'                 # 'live' atau 'demo' (demo bersifat SIMULATED)
MODEL_IDS = ['Qwen/Qwen2.5-0.5B-Instruct']
MAX_CASES = 8                    # 8 untuk smoke test, 40 untuk benchmark penuh
MAX_NEW_TOKENS = 280
DO_SAMPLE = False                # deterministic generation
SEED = 42
OUTPUT_DIR = Path('/content/turing_stress_outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slower)')
except ImportError: print('PyTorch unavailable')

## 3. Memuat 40 benchmark cases
Jika file `benchmark_40.json` tidak ditemukan, notebook meminta upload lewat Colab. Saat memakai paket ZIP, unggah `benchmark_40.json` dari paket ini. Untuk notebook yang berdiri sendiri, dataset juga ditempel di akhir notebook sebagai fallback. Anda juga bisa menggunakan file `benchmark_40.csv` di luar notebook untuk audit.

In [ ]:
BENCHMARK_CASES = json.loads(r'''[{"id": "SE-01", "domain": "Software Engineering", "title": "Idempotent payment retry", "prompt": "A POST /charge endpoint has an idempotency key. Client times out after the payment succeeds and retries with the same key but a different amount. Specify safe server behavior and test cases.", "expected_answer": "Persist the initial key/request fingerprint and resulting outcome atomically. Identical retry returns the previous outcome without charging again; conflicting payload with same key must fail (e.g., 409). Test timeout, concurrent requests, retries, key expiry, failure states.", "required_patterns": ["idempoten", "(same|identical|original|previous|cached|prior).{0,55}(result|response|outcome)|return.{0,35}(original|previous|same)", "(conflict|409|reject|mismatch|different payload)", "(concurren|race|timeout|retry)"], "prohibited_patterns": ["(charge|debit).{0,30}(twice|again|second time)"], "primary_error_category": "Concurrency & Idempotency", "severity": "Critical", "recommended_fix": "Add atomic key+payload fingerprint storage and concurrent retry tests."}, {"id": "SE-02", "domain": "Software Engineering", "title": "Optimistic locking conflict", "prompt": "Two users read order version 4. Both edit quantity and submit updates with If-Match: 4. What must happen and how should this be tested?", "expected_answer": "Only one update should succeed and increment the version. The stale update returns 409/412 and is not applied; client must re-fetch and explicitly resolve the conflict. Test race, retry, and no lost update.", "required_patterns": ["(one|first|only).{0,45}(succeed|success|accept|commit)", "(409|412|conflict|precondition)", "(stale|version|etag)", "(lost update|overwrite|re.?fetch|reload|merge)"], "prohibited_patterns": ["both.{0,30}(succeed|success|commit)"], "primary_error_category": "Concurrency & Data Integrity", "severity": "High", "recommended_fix": "Use version checks in the write transaction and test stale writes."}, {"id": "SE-03", "domain": "Software Engineering", "title": "Cache invalidation after delete", "prompt": "An API deletes a customer, but a read-through cache still serves the customer for 10 minutes. Provide a design and an acceptance test to prevent stale reads.", "expected_answer": "Invalidate or update cached customer entries when deletion commits, ideally via transactional outbox/event with reliable delivery; until consistent, use tombstones or version checks. Verify read-after-delete and failure/retry behavior.", "required_patterns": ["(invalidat|evict|tombstone|version)", "(delet|commit)", "(test|verify|assert|check)", "(fail|retry|race|event|outbox|consisten)"], "prohibited_patterns": ["(wait|sleep).{0,30}(10 minute|ttl)"], "primary_error_category": "State Consistency", "severity": "High", "recommended_fix": "Add delete-triggered invalidation and tests across cache boundaries."}, {"id": "SE-04", "domain": "Software Engineering", "title": "Webhook at least once", "prompt": "A shipping provider sends webhook events at least once and sometimes out of order. How should the receiver process DELIVERED before IN_TRANSIT and avoid duplicates?", "expected_answer": "Verify signatures, deduplicate by event ID, store events, enforce permitted monotonic state transitions or version/sequence ordering, and safely handle out-of-order delivery. Include duplicate, replay, and ordering tests.", "required_patterns": ["(dedup|idempoten|event.?id)", "(order|sequence|version|monoton)", "(signature|authenticat|verify)", "(replay|duplicate|test)"], "prohibited_patterns": ["blindly apply|always overwrite"], "primary_error_category": "Event Ordering", "severity": "High", "recommended_fix": "Apply idempotent event processing with monotonic transition rules."}, {"id": "SE-05", "domain": "Software Engineering", "title": "PII in application logs", "prompt": "A login service logs full JWTs, email addresses, and passwords on failed authentication. Propose immediate remediation and regression tests.", "expected_answer": "Remove secrets and credentials from logs, redact or hash/minimize PII as appropriate, revoke exposed tokens and rotate credentials where relevant, restrict log access and retention, and test logs for secret patterns.", "required_patterns": ["(redact|mask|remov|saniti|not log)", "(password|jwt|token)", "(revoke|rotat|invalidat)", "(regression|test|scan|assert)"], "prohibited_patterns": ["log.{0,30}(full|raw|plaintext).{0,20}(password|jwt)"], "primary_error_category": "Security & Privacy", "severity": "Critical", "recommended_fix": "Introduce log sanitization, secret scanning, and incident-response steps."}, {"id": "SE-06", "domain": "Software Engineering", "title": "Transactional outbox", "prompt": "Order creation commits in a database but publishing OrderCreated to the message broker fails. Explain a robust architecture and tests.", "expected_answer": "Write order and outbox event in one database transaction; asynchronous publisher retries with deduplication/idempotent consumers. Test broker outage, crash between commit and publish, and duplicate publishes.", "required_patterns": ["outbox", "(same|single|one|atomic).{0,40}transaction", "(retry|publish|poll)", "(idempoten|dedup|duplicate)"], "prohibited_patterns": ["two.?phase commit is required"], "primary_error_category": "Distributed Systems Reliability", "severity": "High", "recommended_fix": "Use the transactional outbox pattern and replay-safe consumers."}, {"id": "SE-07", "domain": "Software Engineering", "title": "Time zone DST booking", "prompt": "A meeting is requested for 01:30 on the day daylight saving time ends in New York. Why is this ambiguous and what API data must be captured?", "expected_answer": "Local 01:30 may occur twice. Store an unambiguous instant with UTC offset/time-zone identifier (IANA zone), explicitly resolve fold/offset; test both occurrences and DST spring gap.", "required_patterns": ["(twice|ambiguous|two instants|fold)", "(offset|utc)", "(time.?zone|iana|america/new_york)", "(test|gap|spring)"], "prohibited_patterns": ["01:30.{0,35}unambiguous"], "primary_error_category": "Temporal Edge Case", "severity": "High", "recommended_fix": "Require zone and disambiguating offset/fold; test DST boundaries."}, {"id": "SE-08", "domain": "Software Engineering", "title": "Feature flag rollback", "prompt": "A new price calculation feature is enabled for 5% of customers and causes wrong totals. Describe safe rollback and validation including already-written orders.", "expected_answer": "Disable flag for new requests, preserve/identify affected orders, reconcile or compensate mispriced transactions, monitor metrics, and test both flag branches and rollout consistency.", "required_patterns": ["(disabl|rollback|turn off)", "(affect|existing|already|historical)", "(reconcil|compensat|correct|refund)", "(monitor|metric|test|validate)"], "prohibited_patterns": ["(rollback|disable).{0,30}(undo|revert).{0,20}all"], "primary_error_category": "Release Safety", "severity": "High", "recommended_fix": "Add remediation for persisted side effects and shadow/canary tests."}, {"id": "SE-09", "domain": "Software Engineering", "title": "API pagination mutation", "prompt": "An API uses offset pagination while new records arrive during traversal. A client observes duplicates or missing entries. Recommend an approach and tests.", "expected_answer": "Use stable ordering with cursor/keyset pagination, deterministic tie-breakers, optionally snapshots for exact repeatable traversal. Test concurrent insert/delete and page boundaries.", "required_patterns": ["(cursor|keyset)", "(stable|deterministic|tie.break|sort)", "(concurren|insert|delet|mutation)", "(duplicat|miss|skip|test)"], "prohibited_patterns": ["offset pagination guarantees"], "primary_error_category": "Data Retrieval Semantics", "severity": "Medium", "recommended_fix": "Switch to stable cursor pagination and test mutation across pages."}, {"id": "SE-10", "domain": "Software Engineering", "title": "Rate limit race", "prompt": "An endpoint allows 100 requests/minute per API key, enforced by counters on three stateless application nodes. How do you prevent limit bypass?", "expected_answer": "Use shared atomic rate-limit state (e.g., Redis script/token bucket) with bounded clock semantics, choose failure policy, and test simultaneous requests across nodes and window boundaries.", "required_patterns": ["(shared|central|redis|distribut)", "(atomic|script|lock|transaction)", "(concurren|simultaneous|race)", "(boundary|window|fail|test)"], "prohibited_patterns": ["local counter.{0,30}(sufficient|enough)"], "primary_error_category": "Concurrency & Rate Limiting", "severity": "High", "recommended_fix": "Use atomic distributed counters and cross-node load tests."}, {"id": "PR-01", "domain": "Programming", "title": "Python mutable default argument", "prompt": "What does this Python code print and why?\n```python\ndef add(x, acc=[]):\n    acc.append(x)\n    return acc\nprint(add(1))\nprint(add(2))\n```\nPropose a correct fix.", "expected_answer": "It prints [1] then [1, 2]; default list is created once and shared between calls. Use acc=None and create a new list inside when None.", "required_patterns": ["\\[1\\]", "\\[1,\\s*2\\]", "(shared|once|reus|persist|mutable default)", "none"], "prohibited_patterns": ["\\[2\\].{0,30}(second|print)"], "primary_error_category": "Language Semantics", "severity": "High", "recommended_fix": "Explain shared default evaluation; replace mutable default with None sentinel."}, {"id": "PR-02", "domain": "Programming", "title": "SQL NULL semantics", "prompt": "For a SQL table t(x) with rows (NULL), (1), (2), what does SELECT COUNT(*) FROM t WHERE x != 1 return? Explain and provide a NULL-inclusive predicate.", "expected_answer": "COUNT is 1 because NULL != 1 is UNKNOWN and filtered out; NULL-inclusive predicate is x <> 1 OR x IS NULL.", "required_patterns": ["\\b1\\b", "(null|unknown|three.value)", "(is null)", "(or|\\|\\|)"], "prohibited_patterns": ["(count|return).{0,12}\\b2\\b"], "primary_error_category": "SQL Semantics", "severity": "High", "recommended_fix": "Add SQL three-valued logic examples and test NULL records."}, {"id": "PR-03", "domain": "Programming", "title": "JavaScript asynchronous ordering", "prompt": "Predict output of:\n```js\nconsole.log('A');\nsetTimeout(() => console.log('B'), 0);\nPromise.resolve().then(() => console.log('C'));\nconsole.log('D');\n```\nExplain queue behavior.", "expected_answer": "Output A, D, C, B: synchronous code first, microtask (Promise) before timer macrotask.", "required_patterns": ["A.{0,30}D.{0,30}C.{0,30}B", "(microtask|promise)", "(macrotask|timer|task queue)"], "prohibited_patterns": ["A.{0,30}B.{0,30}C.{0,30}D"], "primary_error_category": "Async Runtime Reasoning", "severity": "High", "recommended_fix": "Explain sync stack, microtask queue, and timer scheduling."}, {"id": "PR-04", "domain": "Programming", "title": "Integer overflow average", "prompt": "In a language with signed 32-bit integers, why can mid=(low+high)/2 overflow during binary search? Provide a safer expression and caveat.", "expected_answer": "low+high can overflow. Use low + (high-low)/2 when 0<=low<=high and subtraction is representable; account for language-specific integer overflow and indexing constraints.", "required_patterns": ["overflow", "low\\s*\\+\\s*\\(?\\s*high\\s*-\\s*low", "(integer|32.bit|range|bounds|sign|caveat)"], "prohibited_patterns": ["(low\\s*\\+\\s*high)\\s*/\\s*2.{0,30}(safe|never)"], "primary_error_category": "Numeric Edge Case", "severity": "High", "recommended_fix": "Use overflow-aware midpoint and boundary tests."}, {"id": "PR-05", "domain": "Programming", "title": "Python exception finally return", "prompt": "Predict output for:\n```python\ndef f():\n    try:\n        return 'try'\n    finally:\n        return 'finally'\nprint(f())\n```\nExplain the control flow and risk.", "expected_answer": "It prints finally. A return in finally overrides the pending return (and can suppress exceptions). Avoid returning from finally.", "required_patterns": ["finally", "(overrid|replac|supersed|suppress)", "(exception|return|risk)"], "prohibited_patterns": ["prints? .?try.?"], "primary_error_category": "Control Flow", "severity": "High", "recommended_fix": "Explain finally override and avoid control-flow exit from finally."}, {"id": "PR-06", "domain": "Programming", "title": "Unicode string length", "prompt": "Explain why length of the string '🇮🇩' can differ across Python len(), JavaScript .length, and user-visible characters. Give correct handling advice.", "expected_answer": "It is one grapheme cluster, built from two regional indicator code points; Python len is 2, JavaScript UTF-16 .length is 4. User-perceived character count requires grapheme segmentation.", "required_patterns": ["(grapheme|user.perceived)", "(python|code.point).{0,35}\\b2\\b|\\b2\\b.{0,35}python", "(javascript|utf.?16).{0,35}\\b4\\b|\\b4\\b.{0,35}javascript", "(segment|grapheme cluster)"], "prohibited_patterns": ["(python.{0,20}1|javascript.{0,20}2)"], "primary_error_category": "Unicode Semantics", "severity": "Medium", "recommended_fix": "Differentiate code units, code points, and grapheme clusters."}, {"id": "PR-07", "domain": "Programming", "title": "SQL aggregation duplicate join", "prompt": "Orders(id,total) joins OrderItems(order_id,...). Why does SUM(Orders.total) over the join overstate revenue? Provide a valid query strategy.", "expected_answer": "One order is repeated per item, inflating SUM. Sum directly from Orders, or aggregate at one row per order before summing; use distinct cautiously because same totals may repeat across different orders.", "required_patterns": ["(duplicat|repeat|multipli|fan.out)", "(sum|aggregat)", "(orders|per.order|subquery|pre.aggregat)", "(distinct|same total|pitfall|caution)"], "prohibited_patterns": ["sum\\s*\\(\\s*distinct\\s+total\\s*\\).{0,35}(always|correct)"], "primary_error_category": "SQL Aggregation", "severity": "High", "recommended_fix": "Aggregate at the correct grain and test orders with multiple items."}, {"id": "PR-08", "domain": "Programming", "title": "Floating point equality", "prompt": "What happens in Python for 0.1 + 0.2 == 0.3? Explain and recommend numeric handling for money.", "expected_answer": "Usually False due to binary floating-point representation. Use math.isclose for approximate comparisons, Decimal or integer minor units for money with defined rounding.", "required_patterns": ["false", "(binary|float|represent|precision)", "(decimal|integer|cents|minor unit)", "(isclose|tolerance|round)"], "prohibited_patterns": ["(true|exact).{0,20}(because|always)"], "primary_error_category": "Numeric Precision", "severity": "High", "recommended_fix": "Explain binary representation and use decimal/integer money semantics."}, {"id": "PR-09", "domain": "Programming", "title": "Path traversal mitigation", "prompt": "A web service reads files under /srv/uploads using a user-provided filename. Is concatenating the path safe? Show design for rejecting ../ and symlink escapes.", "expected_answer": "Do not simply concatenate; canonicalize/resolve path and ensure it stays within root, check symlinks/races where applicable, preferably use opaque IDs and secure openat-style APIs with constrained permissions.", "required_patterns": ["(traversal|\\.\\.|escape)", "(resolv|canonical|realpath|normalize)", "(root|base|allowlist|directory)", "(symlink|race|opaque|openat)"], "prohibited_patterns": ["(strip|replace).{0,20}\\.\\..{0,20}(sufficient|enough)"], "primary_error_category": "Security Vulnerability", "severity": "Critical", "recommended_fix": "Use root-constrained path resolution and race-resistant file opening."}, {"id": "PR-10", "domain": "Programming", "title": "Python dict concurrent mutation", "prompt": "What happens when deleting dictionary entries during iteration in Python? Give a safe approach and explain its complexity trade-off.", "expected_answer": "Mutating dict size during active iteration raises RuntimeError; iterate over list(d) or list(d.items()) snapshot or build new dict; extra O(n) memory for snapshot.", "required_patterns": ["runtimeerror|changed size during iteration", "(list\\(|copy|snapshot|comprehension)", "(memory|space|o\\(n\\)|overhead)"], "prohibited_patterns": ["(safe|allowed).{0,30}delet.{0,20}during iteration"], "primary_error_category": "Collection Semantics", "severity": "Medium", "recommended_fix": "Iterate over snapshot or create filtered dictionary; discuss overhead."}, {"id": "ERP-01", "domain": "ERP", "title": "Inventory FIFO negative stock", "prompt": "ERP warehouse allows stock to go negative on Monday; physical receipt arrives Wednesday at a new unit cost. What risks exist for FIFO valuation and COGS, and which controls/reconciliation tests are needed?", "expected_answer": "Negative stock disrupts FIFO layers/cost timing and can create revaluation or provisional COGS, depending on ERP policy. Recommend prevent/flag negative stock, perform cost adjustment/reconciliation, test backdated movements and GL tie-out; avoid claiming universal product behavior.", "required_patterns": ["(negative stock|negative inventory)", "(fifo|layer|valuation|cogs)", "(revalu|adjust|reconcil|provisional)", "(gl|ledger|backdat|test)"], "prohibited_patterns": ["negative stock.{0,35}no (impact|effect)"], "primary_error_category": "ERP Inventory Costing", "severity": "High", "recommended_fix": "Specify product-specific negative stock policy and valuation reconciliation."}, {"id": "ERP-02", "domain": "ERP", "title": "Intercompany elimination", "prompt": "A group sells inventory from Entity A to Entity B with a 20% markup; B still holds it at period-end. How must consolidated reporting treat sales and unrealized profit?", "expected_answer": "Eliminate intercompany receivable/payable and intercompany revenue/COGS; eliminate unrealized profit in closing inventory on consolidation (compute only if base and markup basis supplied). Test elimination journals and currency consistency.", "required_patterns": ["(eliminat|reverse)", "(intercompany|intra.group)", "(unrealiz|inventory profit|profit in inventory)", "(consolidat|group)"], "prohibited_patterns": ["(recogniz|keep).{0,25}(all|full).{0,20}profit"], "primary_error_category": "ERP Consolidation", "severity": "Critical", "recommended_fix": "Add elimination entries for intra-group balances, sales and unrealized inventory profit."}, {"id": "ERP-03", "domain": "ERP", "title": "Three-way match tolerance", "prompt": "PO quantity 100 at $10, goods receipt 95, supplier invoice 100 at $11. Design an AP three-way match workflow; do not invent tolerance policy.", "expected_answer": "Compare PO, receipt, invoice: qty variance of 5 invoiced units vs received, price variance $1/unit vs PO. Hold/route for approval according to configured tolerances; do not auto-post without policy. Clarify partial receipt and tax/freight.", "required_patterns": ["(three.way|po|purchase order)", "(95|5 units|quantity variance|qty variance)", "(11|1/unit|price variance|price difference)", "(tolerance|approval|hold|policy)"], "prohibited_patterns": ["auto.?approve.{0,20}without"], "primary_error_category": "ERP AP Controls", "severity": "High", "recommended_fix": "Separate quantity/price variances and require configured approvals."}, {"id": "ERP-04", "domain": "ERP", "title": "Multi-currency realized FX", "prompt": "A USD customer invoice is booked in IDR at one rate and settled at a different rate. What accounting entries and tests are needed? Avoid assuming tax rules.", "expected_answer": "Recognize AR/revenue at invoice-date functional-currency amount; settlement clears AR, records cash and realized FX gain/loss for rate difference; reconcile AR aging and bank, and handle period-end unrealized FX separately.", "required_patterns": ["(ar|receivable)", "(realiz|exchange|fx|forex)", "(gain|loss|difference)", "(reconcil|settle|bank|period)"], "prohibited_patterns": ["(same|identical).{0,20}exchange rate"], "primary_error_category": "ERP Accounting & FX", "severity": "High", "recommended_fix": "Separate invoice, settlement, revaluation and realized FX postings."}, {"id": "ERP-05", "domain": "ERP", "title": "Serialized equipment traceability", "prompt": "An ERP distributor sells equipment with serial number S123. A warranty recall affects only a component lot installed in some serials. Define data linkage and test cases.", "expected_answer": "Link finished serial to component lot/serial and work/order trace, shipment/customer, and service/warranty history; support forward and backward traceability. Tests include multi-lot assembly, replacement components and recall selection accuracy.", "required_patterns": ["(serial|s123)", "(lot|batch|component)", "(trace|genealog)", "(customer|shipment|warranty|service)"], "prohibited_patterns": ["recall.{0,20}all equipment"], "primary_error_category": "ERP Traceability", "severity": "High", "recommended_fix": "Build serial-to-component genealogy and targeted recall queries."}, {"id": "ERP-06", "domain": "ERP", "title": "Stock transfer in transit", "prompt": "Company transfers inventory from Site A to Site B using in-transit location. Truck departed but receiving is incomplete at month-end. What does reconciliation need to show?", "expected_answer": "A issues inventory to in-transit, B receives only on confirmation; recognize in-transit ownership/value per accounting policy, prevent double count, reconcile shipment-receipt quantities and GL at cut-off.", "required_patterns": ["in.transit", "(site a|site b|ship|receiv)", "(cut.off|month.end|period)", "(reconcil|double.count|gl|ledger)"], "prohibited_patterns": ["(both sites|site a and site b).{0,35}own.{0,20}same"], "primary_error_category": "ERP Inventory Cut-off", "severity": "High", "recommended_fix": "Reconcile transfer states and ownership for period-end close."}, {"id": "ERP-07", "domain": "ERP", "title": "Segregation of duties", "prompt": "In an ERP, the same user can create vendors, change bank details and release payments. Identify the SoD risk and mitigation/monitoring controls.", "expected_answer": "High fraud risk: fake vendor/payment diversion. Separate vendor creation, bank amendment and payment approval; dual authorization, bank change verification, audit logs and exception monitoring. Test role combinations.", "required_patterns": ["(segregat|sod|separat)", "(vendor|bank)", "(approval|dual|four.eyes)", "(audit|monitor|log|test)"], "prohibited_patterns": ["same user.{0,30}acceptable"], "primary_error_category": "ERP Access Controls", "severity": "Critical", "recommended_fix": "Implement role segregation, dual approval and audit monitoring."}, {"id": "ERP-08", "domain": "ERP", "title": "Production backflush variance", "prompt": "A manufacturing ERP backflushes components from BOM standard consumption. Actual yield falls and scrap rises. What inventory and costing exceptions should be tested?", "expected_answer": "Backflush may differ from actual usage; account for scrap, yield, BOM/recipe variance, WIP and material consumption; reconcile physical inventory and cost variances with controls for negative stock and substitutions.", "required_patterns": ["(backflush|bom)", "(scrap|yield)", "(varianc|cost|wip)", "(reconcil|physical|inventory|test)"], "prohibited_patterns": ["(actual|physical).{0,25}always equal"], "primary_error_category": "ERP Manufacturing Cost", "severity": "High", "recommended_fix": "Track scrap, actual consumption and material/WIP variances."}, {"id": "ERP-09", "domain": "ERP", "title": "Credit note and tax reversal", "prompt": "An invoice was posted and partially paid. Customer returns part of the goods. Outline credit note and allocation flow without assuming local tax rules.", "expected_answer": "Create credit note referencing invoice/return, reverse relevant revenue/AR and tax as allowed by jurisdiction, match against open balance or refund; preserve audit trail and test partial payments and multiple returns.", "required_patterns": ["(credit note|credit memo)", "(partial|balance|refund|allocat)", "(tax|jurisdiction|local rule)", "(audit|trace|reference|return)"], "prohibited_patterns": ["(always|universally).{0,20}(tax|vat)"], "primary_error_category": "ERP AR & Compliance", "severity": "High", "recommended_fix": "Use jurisdiction-specific tax rules and test partial settlement allocations."}, {"id": "ERP-10", "domain": "ERP", "title": "Master data duplicate customer", "prompt": "Customer master data includes ACME PT, PT ACME and Acme Indonesia with differing tax IDs. Should they be merged automatically? Design matching and governance workflow.", "expected_answer": "Do not auto-merge using name alone. Use tax/legal IDs, address and external identifiers, confidence/review, stewardship, approval and reversibility, and test false positives/negatives.", "required_patterns": ["(not|avoid|never).{0,20}(auto.?merge|merge automatically)|manual review", "(tax.?id|legal.?id|identif)", "(steward|approv|review|govern)", "(false positive|false negative|revers|test)"], "prohibited_patterns": ["merge.{0,30}(all|automatically).{0,15}(same|similar).{0,15}name"], "primary_error_category": "ERP Master Data Quality", "severity": "High", "recommended_fix": "Require identity evidence and human-approved duplicate resolution."}, {"id": "RT-01", "domain": "Requirements Testing", "title": "Ambiguous latency requirement", "prompt": "Requirement: 'The search page should be fast.' Rewrite as measurable, testable acceptance criteria and identify assumptions requiring stakeholder confirmation.", "expected_answer": "Define measurable threshold (e.g., p95 <= 2 seconds under agreed load, hardware, dataset, network), measurement method and exclusions. Values are illustrative and must be confirmed with stakeholders.", "required_patterns": ["(p95|percentile|latency|response time)", "(load|concurren|network|environment|dataset)", "(measur|test)", "(confirm|stakeholder|assum|agree)"], "prohibited_patterns": ["fast is measurable"], "primary_error_category": "Ambiguity & Testability", "severity": "Medium", "recommended_fix": "Specify performance SLO, workload and measurement conditions."}, {"id": "RT-02", "domain": "Requirements Testing", "title": "Password reset token reuse", "prompt": "Requirement: 'Users can reset password by email.' Write negative/security test scenarios for token expiry, replay and account enumeration.", "expected_answer": "Tests: valid one-time token; expired, already-used and tampered tokens rejected; token tied to account; rate limit; generic response avoids enumeration; invalidate old sessions if policy requires.", "required_patterns": ["(expir|expired)", "(replay|one.time|used token|reuse)", "(enumerat|generic response|disclos)", "(tamper|rate.limit|invalid|test)"], "prohibited_patterns": ["token.{0,25}reusab"], "primary_error_category": "Security Test Coverage", "severity": "Critical", "recommended_fix": "Add token lifecycle, brute-force and enumeration acceptance criteria."}, {"id": "RT-03", "domain": "Requirements Testing", "title": "Conflicting discount rules", "prompt": "Requirements: 'Gold customers receive 10% discount' and 'Promo code SAVE15 grants 15%'. No stacking policy specified. Produce tests without inventing business policy.", "expected_answer": "Flag ambiguity about stacking, precedence and eligibility; create decision table for gold/non-gold x promo valid/invalid, mark combined outcome TBD until stakeholder resolves.", "required_patterns": ["(ambigu|clarif|tbd|unknown|unspecified)", "(stack|preceden|combin)", "(decision table|test case|matrix)", "(stakeholder|business rule|confirm|policy)"], "prohibited_patterns": ["25% discount.{0,20}(must|always|should)"], "primary_error_category": "Contradictory Requirements", "severity": "High", "recommended_fix": "Resolve stacking and priority rules via decision table approval."}, {"id": "RT-04", "domain": "Requirements Testing", "title": "Boundary value age eligibility", "prompt": "Requirement: applicant must be at least 18 years old on application date. List boundary tests, including leap-day birthday and timezone treatment.", "expected_answer": "Test just below 18, exactly 18, just above; leap-day birthdays by jurisdiction policy; define application date/time zone and calendar age calculation, not fixed 365*18 days.", "required_patterns": ["(17|below|just under)", "(18|exact)", "(leap|feb.?29)", "(time.?zone|calendar|date)"], "prohibited_patterns": ["365\\s*\\*\\s*18"], "primary_error_category": "Boundary Conditions", "severity": "High", "recommended_fix": "Use calendar dates and clarify leap-day/time-zone policy."}, {"id": "RT-05", "domain": "Requirements Testing", "title": "Offline sync conflict", "prompt": "Mobile app allows editing an order offline on two devices and syncing later. Generate acceptance scenarios for edit conflicts and retry.", "expected_answer": "Define conflict policy (versioning, manual merge or server precedence by agreement), ensure no silent lost updates, idempotent retries, timestamp/order uncertainty, and sync status feedback; test concurrent edits.", "required_patterns": ["(conflict|version|merge)", "(lost update|overwrite|silent)", "(retry|idempoten|offline)", "(test|scenario|device|sync)"], "prohibited_patterns": ["last writer wins.{0,25}(always|correct|safe)"], "primary_error_category": "State Synchronization Tests", "severity": "High", "recommended_fix": "Specify and test explicit conflict-resolution strategy."}, {"id": "RT-06", "domain": "Requirements Testing", "title": "Refund overpayment invariant", "prompt": "An order costs $100 and has a $60 refund already processed. A second refund request asks for $50. Define invariant and expected behavior under concurrent refund requests.", "expected_answer": "Cumulative successful refunds <= original refundable amount. Remaining maximum $40; reject or cap second $50 per specified policy; enforce atomically under concurrency and test duplicate retries.", "required_patterns": ["(40|remaining)", "(100|original)", "(reject|cap|partial|deny|not allowed)", "(atomic|concurren|race|lock|transaction)"], "prohibited_patterns": ["approve.{0,20}\\$?50"], "primary_error_category": "Financial Invariants", "severity": "Critical", "recommended_fix": "Enforce atomic cumulative-refund invariant and concurrent request tests."}, {"id": "RT-07", "domain": "Requirements Testing", "title": "Missing error handling in use case", "prompt": "Use case: 'User uploads CSV; system imports records.' Identify alternate/exception flows and define test cases without assuming upload size limits.", "expected_answer": "Include malformed CSV, encoding, empty file, missing headers, duplicate rows, partial import transaction strategy, permission, network interruption, and error reporting; upload limits require stakeholder confirmation.", "required_patterns": ["(malform|invalid|csv|encoding)", "(duplicate|header|empty)", "(partial|rollback|transaction|atomic)", "(limit|confirm|unspecified|stakeholder)"], "prohibited_patterns": ["maximum upload.{0,15}\\d+\\s*(mb|gb)"], "primary_error_category": "Missing Alternate Flows", "severity": "High", "recommended_fix": "Derive negative paths and clarify import atomicity and limits."}, {"id": "RT-08", "domain": "Requirements Testing", "title": "Non-deterministic external payment", "prompt": "Acceptance test triggers external payment gateway. How should test design avoid flaky dependence on live network while still validating integration?", "expected_answer": "Use mocked/stubbed gateway for deterministic contract tests, provider sandbox for integration, limited monitored end-to-end tests, failure injection, retries, timeouts, and idempotency assertions.", "required_patterns": ["(mock|stub|fake)", "(sandbox|contract)", "(timeout|failure|retry)", "(idempoten|determin|flak|test)"], "prohibited_patterns": ["(always|only).{0,30}production gateway"], "primary_error_category": "Test Environment & Flakiness", "severity": "Medium", "recommended_fix": "Separate unit/contract/sandbox/E2E tiers and inject failures."}, {"id": "RT-09", "domain": "Requirements Testing", "title": "Role-based access matrix", "prompt": "Requirements: Viewer can read invoices, Clerk can create drafts, Manager can approve. Define test matrix for users with multiple roles and direct API access.", "expected_answer": "Create role-action matrix for read/create/approve and deny unauthorized operations, evaluate combined-role policy explicitly, test direct API enforcement and privilege escalation rather than only hiding UI buttons.", "required_patterns": ["(matrix|role|permission)", "(viewer|clerk|manager)", "(api|server|endpoint)", "(deny|unauthoriz|escalat|negative)"], "prohibited_patterns": ["hide.{0,15}button.{0,20}sufficient"], "primary_error_category": "Authorization Coverage", "severity": "Critical", "recommended_fix": "Enforce authorization server-side and test role combinations."}, {"id": "RT-10", "domain": "Requirements Testing", "title": "Data retention contradiction", "prompt": "Requirement A: delete customer data within 30 days of request. Requirement B: retain financial records for seven years. How should a tester address this conflict?", "expected_answer": "Identify data categories and legal basis, distinguish deletion from retention/exceptions, seek policy/legal clarification, design anonymization/restriction and audit evidence tests; do not assume a single retention rule applies universally.", "required_patterns": ["(conflict|except|clarif|different)", "(financial|record|retention)", "(delet|anonym|restrict)", "(legal|compliance|policy|stakeholder)"], "prohibited_patterns": ["(delet|purge).{0,35}all.{0,25}30 days"], "primary_error_category": "Compliance Ambiguity", "severity": "High", "recommended_fix": "Clarify data-class retention, lawful exceptions, deletion proof and audit trails."}]''')
assert len(BENCHMARK_CASES) == 40
assert len({x['id'] for x in BENCHMARK_CASES}) == 40
case_df = pd.DataFrame(BENCHMARK_CASES)
print('Cases:',len(case_df))
display(case_df.groupby('domain').size().rename('n_cases').reset_index())
display(case_df[['id','domain','title','severity','primary_error_category']].head(12))
with open(OUTPUT_DIR/'benchmark_40.json','w',encoding='utf-8') as f: json.dump(BENCHMARK_CASES,f,ensure_ascii=False,indent=2)
case_df.to_csv(OUTPUT_DIR/'benchmark_40.csv',index=False,encoding='utf-8-sig')

## 4. Periksa contoh prompt & expected answer

In [ ]:
selected = BENCHMARK_CASES[0]
print('CASE:',selected['id'], '-', selected['title'])
print('\nPROMPT:\n',selected['prompt'])
print('\nEXPECTED ANSWER:\n',selected['expected_answer'])
print('\nERROR CLASS:',selected['primary_error_category'])
print('\nRECOMMENDED FIX:',selected['recommended_fix'])

## 5. Jalankan inferensi LLM (model.generate, bukan pipeline)
Prompt bahasa Inggris disiapkan untuk mensimulasikan evaluasi profesional. Model dijalankan dalam mode deterministik (`do_sample=False`). Dataset benchmark disimpan terpisah dari respons agar tidak mengotori evaluation reference.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

def build_messages(case):
    system = ("You are a careful software engineering reviewer. Answer the user's question accurately. "
              "Do not invent missing business rules, laws, model-specific capabilities or undocumented assumptions. "
              "State uncertainty clearly and provide concrete edge cases when requested. Keep your answer concise.")
    return [{'role':'system','content':system}, {'role':'user','content':case['prompt']}]

def run_one_model(model_id, selected_cases):
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    kwargs = {'torch_dtype':dtype, 'trust_remote_code':False, 'low_cpu_mem_usage':True}
    if torch.cuda.is_available(): kwargs['device_map'] = 'auto'
    model = AutoModelForCausalLM.from_pretrained(model_id, **kwargs)
    if not torch.cuda.is_available(): model.to('cpu')
    model.eval()
    results=[]
    for i,case in enumerate(selected_cases,1):
        try:
            messages=build_messages(case)
            prompt=tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
            inputs=tokenizer(prompt, return_tensors='pt',truncation=True,max_length=3072)
            device = next(model.parameters()).device
            inputs={k:v.to(device) for k,v in inputs.items()}
            start=time.perf_counter()
            with torch.inference_mode():
                ids=model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=DO_SAMPLE,
                                   pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
            response=tokenizer.decode(ids[0][inputs['input_ids'].shape[-1]:],skip_special_tokens=True).strip()
            latency=round(time.perf_counter()-start,2)
            results.append({'id':case['id'],'model_id':model_id,'model_response':response,
                            'latency_seconds':latency,'response_source':'LIVE_MODEL','inference_error':''})
            print(f"{i:02d}/{len(selected_cases):02d}  {case['id']}  {latency}s")
        except Exception as ex:
            print('ERROR',case['id'],type(ex).__name__,str(ex)[:160])
            results.append({'id':case['id'],'model_id':model_id,'model_response':'',
                            'latency_seconds':None,'response_source':'LIVE_MODEL','inference_error':str(ex)[:300]})
    del model,tokenizer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return results

selected_cases = [case for j in range(10) for case in [BENCHMARK_CASES[j], BENCHMARK_CASES[10+j], BENCHMARK_CASES[20+j], BENCHMARK_CASES[30+j]]][:MAX_CASES]  # round-robin: all domains represented
if RUN_MODE == 'live':
    all_responses=[]
    for model_id in MODEL_IDS:
        print('\nRunning:',model_id)
        all_responses.extend(run_one_model(model_id,selected_cases))
elif RUN_MODE == 'demo':
    all_responses=[{'id':c['id'],'model_id':'SIMULATED_DEMO',
       'model_response':c['expected_answer'] if i%2==0 else 'This is a generic response without adequate technical evidence.',
       'latency_seconds':0,'response_source':'SIMULATED_DEMO','inference_error':''}
       for i,c in enumerate(selected_cases)]
else: raise ValueError('RUN_MODE harus live atau demo')
print('Responses:',len(all_responses))
assert all(r['response_source']=='LIVE_MODEL' for r in all_responses) if RUN_MODE=='live' else True

## 6. Automatic screening — concept coverage, prohibited patterns, error category
Metrik **concept_coverage = matched expected concepts / total concepts**. Ini hanya *screening proxy*, bukan accuracy atau faithfulness metric. `prohibited_pattern_hits` bisa menghasilkan false positive; hasilnya wajib direview secara manual. `inference_error` diberi status `INFERENCE_ERROR` dan **dikeluarkan dari denominator** evaluasi jawaban.

In [ ]:
def screen_response(case, result):
    row={k:case[k] for k in ['id','domain','title','prompt','expected_answer','primary_error_category','severity','recommended_fix']}
    row.update(result)
    answer=result.get('model_response','') or ''
    if result.get('inference_error') or not answer.strip():
        row.update(concept_coverage=np.nan, matched_concepts=0, total_concepts=len(case['required_patterns']),
                   missing_concepts=json.dumps([]), prohibited_pattern_hits=json.dumps([]),
                   screening_flag='INFERENCE_ERROR' if result.get('inference_error') else 'EMPTY_RESPONSE',
                   suspected_error_category='Inference failure' if result.get('inference_error') else 'Empty response',
                   suggested_remediation='Check inference logs and rerun model; do not score this as a model reasoning failure.')
        return row
    hits=[bool(re.search(p,answer,flags=re.I|re.S)) for p in case['required_patterns']]
    missing=[p for p,hit in zip(case['required_patterns'],hits) if not hit]
    banned=[p for p in case['prohibited_patterns'] if re.search(p,answer,flags=re.I|re.S)]
    cov=sum(hits)/len(hits) if hits else np.nan
    flag = 'REVIEW_PROHIBITED_CLAIM' if banned else ('REVIEW_MISSING_CONCEPTS' if missing else 'NO_HEURISTIC_FLAG')
    row.update(concept_coverage=round(cov,3),matched_concepts=sum(hits),total_concepts=len(hits),
               missing_concepts=json.dumps(missing),prohibited_pattern_hits=json.dumps(banned),
               screening_flag=flag,
               suspected_error_category=case['primary_error_category'] if (banned or missing) else 'None flagged',
               suggested_remediation=case['recommended_fix'] if (banned or missing) else 'Perform human factual and reasoning review.')
    return row

case_by_id={c['id']:c for c in BENCHMARK_CASES}
scored=pd.DataFrame([screen_response(case_by_id[r['id']],r) for r in all_responses])
print('Screening rows:',len(scored))
display(scored[['id','model_id','concept_coverage','screening_flag','suspected_error_category','response_source']])
scored.to_csv(OUTPUT_DIR/'screening_results.csv',index=False,encoding='utf-8-sig')

## 7. Uji semantic similarity (opsional, lexical TF-IDF)
Nilai ini **tidak** dijadikan factual accuracy. Cocok untuk menilai kesamaan teks sebagai indikator sekunder. Nilai rendah dapat terjadi pada parafrase yang benar.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def lexical_reference_similarity(row):
    response=row['model_response'] or ''
    if not response.strip(): return np.nan
    docs=[row['expected_answer'],response]
    try:
        m=TfidfVectorizer(ngram_range=(1,2),stop_words='english').fit_transform(docs)
        return round(float(cosine_similarity(m[0],m[1])[0,0]),3)
    except ValueError: return np.nan
scored['reference_tfidf_similarity']=scored.apply(lexical_reference_similarity,axis=1)
display(scored[['id','concept_coverage','reference_tfidf_similarity']])

## 8. Human evaluation template
Human reviewer menilai **factual accuracy, reasoning, relevance, completeness (1–5)** dan label **confirmed_error_category**. Manual review harus menggunakan expected answer, domain knowledge, dan evidence tambahan jika perlu. Kosong/NaN berarti **BELUM DINILAI**, bukan 0. Edit CSV di Colab/Sheets kemudian upload pada sel berikut.

In [ ]:
review_template=scored[['id','model_id','domain','title','prompt','expected_answer','model_response',
                              'screening_flag','suspected_error_category','recommended_fix']].copy()
for name in ['factual_accuracy_1_5','reasoning_1_5','relevance_1_5','completeness_1_5']:
    review_template[name]=np.nan
review_template['confirmed_error_category']=''
review_template['reviewer_comments']=''
review_template['human_verified']=False
review_path=OUTPUT_DIR/'human_review_template.csv'
review_template.to_csv(review_path,index=False,encoding='utf-8-sig')
print('Saved:',review_path)
display(review_template.head(3))

### Optional: upload reviewed CSV
Set `UPLOAD_REVIEWED_CSV=True` dan pilih file yang sudah diisi. Kalau tidak, human metrics tetap kosong dan dashboard tidak mengklaim skor kualitas model.

In [ ]:
UPLOAD_REVIEWED_CSV=False
human_review=None
if UPLOAD_REVIEWED_CSV:
    from google.colab import files
    uploaded=files.upload()
    if not uploaded: raise RuntimeError('No review file selected')
    filename=next(iter(uploaded))
    human_review=pd.read_csv(filename)
    required={'id','model_id','factual_accuracy_1_5','reasoning_1_5','relevance_1_5','completeness_1_5'}
    if not required.issubset(human_review.columns): raise ValueError(f'Missing columns: {required-set(human_review.columns)}')
    for key in ['factual_accuracy_1_5','reasoning_1_5','relevance_1_5','completeness_1_5']:
        human_review[key]=pd.to_numeric(human_review[key],errors='coerce')
        if not (human_review[key].dropna().between(1,5)).all(): raise ValueError(f'{key} must be 1..5 or empty')
    human_review=human_review.drop_duplicates(['id','model_id'],keep='last')
    print('Loaded human-reviewed rows:',len(human_review))
else:
    print('No human review uploaded. Human accuracy/reasoning metrics will remain unavailable.')

## 9. Agregasi dan visualisasi (denominator eksplisit)
Grafik screening menampilkan **heuristic concept coverage** dan proporsi flag yang memerlukan review, bukan akurasi atau hallucination rate. Jika human review tersedia, tampilkan mean factual accuracy, reasoning, relevance dan completeness khusus baris yang benar-benar direview.

In [ ]:
valid=scored[~scored['screening_flag'].isin(['INFERENCE_ERROR','EMPTY_RESPONSE'])].copy()
if valid.empty:
    print('Tidak ada respons valid untuk digrafikkan.')
else:
    summary=(valid.groupby(['model_id','domain'],as_index=False)
             .agg(n_responses=('id','size'),mean_concept_coverage=('concept_coverage','mean'),
                  flagged_for_review=('screening_flag',lambda s:int((s!='NO_HEURISTIC_FLAG').sum()))))
    summary['flagged_review_rate']=summary['flagged_for_review']/summary['n_responses']
    display(summary)
    summary.to_csv(OUTPUT_DIR/'model_domain_summary.csv',index=False)
    plt.figure(figsize=(11,4.5))
    for model_name,part in summary.groupby('model_id'):
        plt.plot(part['domain'],part['mean_concept_coverage'],marker='o',label=model_name)
    plt.ylim(0,1.05);plt.ylabel('Mean concept coverage (heuristic)');plt.title('Benchmark concept-coverage screening by domain')
    plt.xticks(rotation=15,ha='right');plt.legend();plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'concept_coverage_by_domain.png',dpi=160);plt.show()
    plt.figure(figsize=(10,4))
    rates=valid.groupby('domain')['screening_flag'].apply(lambda s:(s!='NO_HEURISTIC_FLAG').mean())
    rates.plot.bar();plt.ylim(0,1);plt.ylabel('Fraction flagged for manual review')
    plt.title('Potential review flags by domain (NOT hallucination rate)')
    plt.xticks(rotation=15,ha='right');plt.tight_layout()
    plt.savefig(OUTPUT_DIR/'review_flags_by_domain.png',dpi=160);plt.show()

if human_review is not None:
    metrics=['factual_accuracy_1_5','reasoning_1_5','relevance_1_5','completeness_1_5']
    merged=scored.merge(human_review[['id','model_id']+metrics+(['confirmed_error_category'] if 'confirmed_error_category' in human_review else [])],on=['id','model_id'],how='left',validate='one_to_one')
    reviewed=merged[merged[metrics].notna().all(axis=1) & ~merged['screening_flag'].isin(['INFERENCE_ERROR','EMPTY_RESPONSE'])].copy()
    print('Fully reviewed:',len(reviewed),'/',len(scored))
    if not reviewed.empty:
        human_summary=reviewed.groupby('model_id')[metrics].agg(['mean','count'])
        display(human_summary)
        human_summary.to_csv(OUTPUT_DIR/'human_evaluation_summary.csv')
        reviewed.to_csv(OUTPUT_DIR/'human_verified_results.csv',index=False,encoding='utf-8-sig')

## 10. Ekspor CSV / JSON / ZIP
Semua respons model, expected answer, failure taxonomy dan rekomendasi disertakan; tidak ada claim tentang benchmark accuracy tanpa human review.

In [ ]:
scored.to_csv(OUTPUT_DIR/'screening_results.csv',index=False,encoding='utf-8-sig')
scored.replace({np.nan:None}).to_json(OUTPUT_DIR/'screening_results.json',orient='records',force_ascii=False,indent=2)
metadata={'project':'Turing LLM Stress Testing & Edge Case Benchmark','run_mode':RUN_MODE,
          'models':MODEL_IDS if RUN_MODE=='live' else ['SIMULATED_DEMO'],
          'n_cases_in_catalog':len(BENCHMARK_CASES),'n_cases_attempted':len(selected_cases),
          'n_results':len(scored),'n_inference_errors':int((scored['screening_flag']=='INFERENCE_ERROR').sum()),
          'note':'Regex concept coverage and prohibited hits are heuristic screening, not automatic fact checking or confirmed hallucination rates.',
          'seed':SEED,'max_new_tokens':MAX_NEW_TOKENS,'sampling':DO_SAMPLE}
(OUTPUT_DIR/'run_metadata.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
zip_path=Path('/content/turing_stress_benchmark_results.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.glob('*'):
        if p.is_file():z.write(p,arcname=p.name)
print('Output:',zip_path)
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Outside Google Colab: download manually at',zip_path)

## 11. Validitas eksperimen & portfolio reporting

- **Test set:** 40 cases, four domains, ten cases each, expected answers written in advance. Keep test prompts independent from model tuning examples.
- **Model response:** Collect at inference time; `response_source=LIVE_MODEL` for actual model responses. Never misrepresent demo responses.
- **Repeatability:** Seed fixed, `do_sample=False`, record model ID, runtime, and settings. Hardware/software variation may still affect reproducibility.
- **Screening:** Regex coverage and prohibited claims identify candidates for review, **not proven failures**. Inspect false positives/negatives.
- **Gold annotation:** At least two independent reviewers per sampled case; record disagreement, adjudication and annotation guidelines. Use confirmed errors for final model failure metrics.
- **Fair comparison:** Same 40 prompts, same maximum response tokens, same instructions and rubric for each model. Add confidence intervals for larger experiments.
- **Recommended improvements:** Build failed-case clusters, revise prompts/context/retrieval/tool integration, then retest on held-out cases. Do not tune to this benchmark and report that as independent generalization.

**Deliverable:** `screening_results.csv/json`, `human_review_template.csv`, `benchmark_40.csv/json`, charts, run metadata, ZIP.